# Experiment P: Verbalizer Ensemble (Seed Averaging)

Train **three verbalizer models** using the best hyperparameters from
Experiment L, each with a **different random seed**. At inference time,
average the sigmoid probabilities from all three models before applying
the decision threshold.

**Approach:**
1. Load best hyperparameters from Exp L (`exp_L_verbalizer_best_params.json`)
2. Train 3 independent `PCLVerbalizer` models with seeds 42, 123, 7
3. Save each model checkpoint
4. Ensemble: `P_ensemble = mean(sigmoid(score_1), sigmoid(score_2), sigmoid(score_3))`
5. Sweep thresholds on val set, evaluate on dev set

**Why seed ensembling helps:**
- Different initialisations explore different loss-surface basins
- Averaging probabilities reduces prediction variance
- No extra hyperparameters — purely exploits the best Exp L config

In [1]:
import os
import sys
import random
import logging
import gc
import json

import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from utils.data import load_data
from utils.split import split_train_val
from utils.pcl_dataset import PCLVerbalizerDataset
from utils.pcl_deberta import PCLVerbalizer
from utils.optim import compute_pos_weight
from utils.training_loop import train_model

DATA_DIR = "../data"
OUT_DIR = "out"
MODEL_OUT_DIR = OUT_DIR
MODEL_NAME = "roberta-large"
MAX_LENGTH = 256
VAL_FRACTION = 0.15
BATCH_SIZE = 32
NUM_EPOCHS = 12
PATIENCE = 4
N_EVAL_STEPS = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXP_NAME = "P_verbalizer_ensemble"

# Ensemble seeds — deliberately spread apart
ENSEMBLE_SEEDS = [42, 123, 7]

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")
os.makedirs(OUT_DIR, exist_ok=True)

if os.getlogin() == "jc4423":
    LOG.info("Running on lab machines, changing caches and model directory")
    os.environ["HF_HOME"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["PIP_CACHE_DIR"] = "/vol/bitbucket/jc4423/.cache/pip"
    os.environ["XDG_CACHE_HOME"] = "/vol/bitbucket/jc4423/.cache"
    MODEL_OUT_DIR = "/vol/bitbucket/jc4423/models/"

/vol/bitbucket/jc4423/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-04 12:23:06,860 INFO:	Device: cuda
2026-03-04 12:23:06,863 INFO:	Running on lab machines, changing caches and model directory


## 1. Data Loading & Best Params

In [2]:
# Use seed 42 for the data split (must match Exp L)
SPLIT_SEED = 42
train_df, dev_df = load_data(DATA_DIR)
train_sub_df, val_sub_df = split_train_val(train_df, val_frac=VAL_FRACTION, seed=SPLIT_SEED)
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
LOG.info(f"Train: {len(train_sub_df)}, Val: {len(val_sub_df)}, Dev: {len(dev_df)}")

# Load best hyperparameters from Experiment L
best_params_path = os.path.join(OUT_DIR, "exp_L_verbalizer_best_params.json")
with open(best_params_path) as f:
    BEST_PARAMS = json.load(f)
print("Best params from Exp L:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")

2026-03-04 12:23:07,020 INFO:	Train/val split: 7118 train, 1257 val (val_frac=0.15)
2026-03-04 12:23:07,021 INFO:	Train, val positive count: 675, 119
2026-03-04 12:23:07,153 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-04 12:23:07,288 INFO:	HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-04 12:23:07,408 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 12:23:07,508 INFO:	HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-04 12:23:07,610 INFO:	HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-04 12:23:07,71

Best params from Exp L:
  lr: 6.977921490278437e-06
  weight_decay: 0.000691662498760998
  head_lr_multiplier: 10
  verbalizer: true_false
  template_idx: 2
  batch_size: 32
  num_epochs: 12
  patience: 4
  best_threshold: 0.275
  warmup_fraction: 0.06
  label_smoothing: 0.0


## 2. Verbalizer & Template Setup

In [3]:
from torch.utils.data import DataLoader

# --- Verbalizer variants (same as Exp L) ---
VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]


def words_to_first_subword_ids(words: list[str]) -> list[int]:
    """Convert words to their first subword token IDs."""
    ids = []
    for w in words:
        subtokens = tokeniser.encode(w, add_special_tokens=False)
        ids.append(subtokens[0])
        n_sub = len(subtokens)
        decoded = tokeniser.decode([subtokens[0]])
        flag = " Single token" if n_sub == 1 else f" {n_sub} subwords, using first ('{decoded}')"
        LOG.info(f"    '{w}' -> id {subtokens[0]}{flag}")
    return ids


# Resolve best verbalizer and template from Exp L params
BEST_VERBALIZER_NAME = BEST_PARAMS["verbalizer"]
BEST_TEMPLATE_IDX = BEST_PARAMS["template_idx"]
BEST_TEMPLATE = TEMPLATE_OPTIONS[BEST_TEMPLATE_IDX]

verb_set = [v for v in VERBALIZER_SETS if v[0] == BEST_VERBALIZER_NAME][0]
LOG.info(f"Best verbalizer: '{BEST_VERBALIZER_NAME}'")
POS_IDS = words_to_first_subword_ids(verb_set[1])
NEG_IDS = words_to_first_subword_ids(verb_set[2])

print(f"\nVerbalizer: {BEST_VERBALIZER_NAME}")
print(f"  pos words: {verb_set[1]} -> ids {POS_IDS}")
print(f"  neg words: {verb_set[2]} -> ids {NEG_IDS}")
print(f"Template [{BEST_TEMPLATE_IDX}]: {BEST_TEMPLATE.format(text='...', mask='[MASK]')}")


def make_verbalizer_dataloaders(
    template: str,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Create DataLoaders with the verbalizer template applied (single [MASK])."""
    def make_ds(df):
        return PCLVerbalizerDataset(
            texts=df["text"].tolist(),
            labels=df["binary_label"].tolist(),
            max_length=MAX_LENGTH,
            tokeniser=tokeniser,
            template=template,
        )

    train_ds = make_ds(train_sub_df)
    val_ds = make_ds(val_sub_df)
    dev_ds = make_ds(dev_df)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, dev_loader

2026-03-04 12:23:07,893 INFO:	Best verbalizer: 'true_false'
2026-03-04 12:23:07,895 INFO:	    'true' -> id 29225 Single token
2026-03-04 12:23:07,895 INFO:	    'false' -> id 22303 Single token



Verbalizer: true_false
  pos words: ['true'] -> ids [29225]
  neg words: ['false'] -> ids [22303]
Template [2]: Is this text talking down to people? "..." Answer: [MASK]


## 3. Train Ensemble Members

Train 3 models with the same hyperparameters but different random seeds.
Each model is saved independently.

In [4]:
def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# Extract fixed HPs from best params
LR = BEST_PARAMS["lr"]
WEIGHT_DECAY = BEST_PARAMS["weight_decay"]
HEAD_LR_MULT = BEST_PARAMS["head_lr_multiplier"]
WARMUP_FRACTION = BEST_PARAMS.get("warmup_fraction", 0.06)
LABEL_SMOOTHING = BEST_PARAMS.get("label_smoothing", 0.0)

print(f"Fixed HPs from Exp L:")
print(f"  lr:               {LR:.2e}")
print(f"  weight_decay:     {WEIGHT_DECAY:.2e}")
print(f"  head_lr_mult:     {HEAD_LR_MULT}")
print(f"  warmup_fraction:  {WARMUP_FRACTION}")
print(f"  label_smoothing:  {LABEL_SMOOTHING}")
print(f"  verbalizer:       {BEST_VERBALIZER_NAME}")
print(f"  template_idx:     {BEST_TEMPLATE_IDX}")
print(f"  ensemble seeds:   {ENSEMBLE_SEEDS}")
print()

Fixed HPs from Exp L:
  lr:               6.98e-06
  weight_decay:     6.92e-04
  head_lr_mult:     10
  warmup_fraction:  0.06
  label_smoothing:  0.0
  verbalizer:       true_false
  template_idx:     2
  ensemble seeds:   [42, 123, 7]



In [5]:
ensemble_results = []

for i, seed in enumerate(ENSEMBLE_SEEDS):
    LOG.info(f"{'='*60}")
    LOG.info(f"[{EXP_NAME}] Training member {i+1}/{len(ENSEMBLE_SEEDS)} (seed={seed})")
    LOG.info(f"{'='*60}")

    set_seed(seed)

    # Recreate dataloaders (shuffle order depends on seed)
    train_loader, val_loader, dev_loader = make_verbalizer_dataloaders(BEST_TEMPLATE)

    model = PCLVerbalizer(
        pos_verbalizer_ids=POS_IDS,
        neg_verbalizer_ids=NEG_IDS,
        model_name=MODEL_NAME,
        gradient_checkpointing=True,
        cache_dir="/vol/bitbucket/jc4423/.cache/huggingface",
    ).to(DEVICE)

    pos_weight = compute_pos_weight(train_sub_df, DEVICE)

    results = train_model(
        model=model, device=DEVICE,
        train_loader=train_loader, val_loader=val_loader, dev_loader=dev_loader,
        pos_weight=pos_weight, lr=LR, weight_decay=WEIGHT_DECAY,
        num_epochs=NUM_EPOCHS, warmup_fraction=WARMUP_FRACTION,
        patience=PATIENCE, head_lr_multiplier=HEAD_LR_MULT,
        label_smoothing=LABEL_SMOOTHING, eval_every_n_steps=N_EVAL_STEPS,
        use_8bit_adam=True,
    )

    # Save this member's checkpoint
    model_path = os.path.join(MODEL_OUT_DIR, f"exp_{EXP_NAME}_member_{i}_seed{seed}.pt")
    torch.save(
        {k: v.cpu() for k, v in model.state_dict().items()},
        model_path,
    )
    LOG.info(f"Saved member {i+1} to {model_path}")

    ensemble_results.append({
        "seed": seed,
        "member_idx": i,
        "best_val_f1": results["best_val_f1"],
        "best_threshold": results["best_threshold"],
        "dev_f1": results["dev_metrics"]["f1"],
        "dev_precision": results["dev_metrics"]["precision"],
        "dev_recall": results["dev_metrics"]["recall"],
        "model_path": model_path,
    })

    LOG.info(f"Member {i+1}: val F1={results['best_val_f1']:.4f}, "
             f"dev F1={results['dev_metrics']['f1']:.4f}")

    del model, train_loader, val_loader, dev_loader
    gc.collect()
    torch.cuda.empty_cache()

# Summary of individual members
print(f"\n{'Member':<8} {'Seed':<6} {'Val F1':<10} {'Dev F1':<10} {'Dev P':<10} {'Dev R':<10}")
print("-" * 54)
for r in ensemble_results:
    print(f"{r['member_idx']+1:<8} {r['seed']:<6} {r['best_val_f1']:<10.4f} "
          f"{r['dev_f1']:<10.4f} {r['dev_precision']:<10.4f} {r['dev_recall']:<10.4f}")

2026-03-04 12:23:07,950 INFO:	============================================================
2026-03-04 12:23:07,950 INFO:	[P_verbalizer_ensemble] Training member 1/3 (seed=42)
2026-03-04 12:23:07,950 INFO:	============================================================


2026-03-04 12:23:09,181 WARNING:	Sample 1307: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,185 WARNING:	Sample 1734: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,190 WARNING:	Sample 2274: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,202 WARNING:	Sample 3753: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,214 WARNING:	Sample 5179: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,218 WARNING:	Sample 5586: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:23:09,226 WARNING:	Sample 6606: [MASK] token was truncated (text may be too long for max_length=256). Falling back to posi

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.54 GiB. GPU 0 has a total capacity of 15.57 GiB of which 258.00 MiB is free. Process 806719 has 790.00 MiB memory in use. Process 43583 has 708.00 MiB memory in use. Process 848158 has 3.39 GiB memory in use. Process 917263 has 4.46 GiB memory in use. Process 925389 has 398.00 MiB memory in use. Process 932630 has 818.00 MiB memory in use. Including non-PyTorch memory, this process has 4.65 GiB memory in use. Of the allocated memory 2.93 GiB is allocated by PyTorch, and 1.43 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 4. Ensemble Inference

Load all 3 models, collect sigmoid probabilities on val and dev sets,
average them, and sweep thresholds.

In [ ]:
@torch.no_grad()
def collect_probs(
    model: PCLVerbalizer,
    device: torch.device,
    dataloader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    """Run inference and return (probs, labels) as numpy arrays."""
    model.eval()
    all_scores, all_labels = [], []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        mask_token_indices = batch.get("mask_token_indices", None)
        if mask_token_indices is not None:
            mask_token_indices = mask_token_indices.to(device)

        scores = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_indices=mask_token_indices,
        ).squeeze(-1)
        all_scores.append(scores.cpu())
        all_labels.append(labels)

    all_scores = torch.cat(all_scores)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_scores).numpy()
    labels = all_labels.long().numpy()
    return probs, labels


def find_best_threshold_from_probs(
    probs: np.ndarray, labels: np.ndarray,
) -> tuple[float, float]:
    """Sweep thresholds to find the one maximising F1."""
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.25, 0.70, 0.025):
        preds = (probs >= thresh).astype(int)
        f1 = float(f1_score(labels, preds, zero_division=0))
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(thresh)
    return best_thresh, best_f1

In [ ]:
# Create deterministic dataloaders for ensemble evaluation
set_seed(SPLIT_SEED)
_, val_loader, dev_loader = make_verbalizer_dataloaders(BEST_TEMPLATE)

# Collect probabilities from each member
val_probs_list: list[np.ndarray] = []
dev_probs_list: list[np.ndarray] = []
val_labels: np.ndarray = np.array([])
dev_labels: np.ndarray = np.array([])

for i, r in enumerate(ensemble_results):
    LOG.info(f"Loading member {i+1} from {r['model_path']}")

    model = PCLVerbalizer(
        pos_verbalizer_ids=POS_IDS,
        neg_verbalizer_ids=NEG_IDS,
        model_name=MODEL_NAME,
    ).to(DEVICE)

    state_dict = torch.load(r["model_path"], map_location=DEVICE)
    model.load_state_dict(state_dict)

    vp, vl = collect_probs(model, DEVICE, val_loader)
    dp, dl = collect_probs(model, DEVICE, dev_loader)

    val_probs_list.append(vp)
    dev_probs_list.append(dp)

    if len(val_labels) == 0:
        val_labels = vl
        dev_labels = dl

    del model, state_dict
    gc.collect()
    torch.cuda.empty_cache()

# Average probabilities across ensemble members
val_probs_ensemble = np.mean(val_probs_list, axis=0)
dev_probs_ensemble = np.mean(dev_probs_list, axis=0)

LOG.info(f"Ensemble probs collected: val={val_probs_ensemble.shape}, dev={dev_probs_ensemble.shape}")

2026-03-04 12:17:07,790 WARNING:	Sample 1307: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,794 WARNING:	Sample 1734: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,799 WARNING:	Sample 2274: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,811 WARNING:	Sample 3753: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,822 WARNING:	Sample 5179: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,825 WARNING:	Sample 5586: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 12:17:07,833 WARNING:	Sample 6606: [MASK] token was truncated (text may be too long for max_length=256). Falling back to posi

NameError: name 'ensemble_results' is not defined

## 5. Results

In [ ]:
# Find best threshold on val set using ensemble probs
best_thresh, val_f1 = find_best_threshold_from_probs(val_probs_ensemble, val_labels)
LOG.info(f"Ensemble optimal threshold: {best_thresh:.3f} (val F1={val_f1:.4f})")

# Evaluate on dev set
dev_preds = (dev_probs_ensemble >= best_thresh).astype(int)
dev_f1 = f1_score(dev_labels, dev_preds, zero_division=0)

print(f"\n{'='*60}")
print(f"{EXP_NAME.upper()} — Dev Set Results (threshold={best_thresh:.3f})")
print(f"{'='*60}")
print(classification_report(dev_labels, dev_preds, target_names=["Non-PCL", "PCL"]))

# Compare individual members vs ensemble
print(f"\n{'Model':<20} {'Val F1':<10} {'Dev F1':<10}")
print("-" * 40)
for r in ensemble_results:
    print(f"Member {r['member_idx']+1} (seed={r['seed']}){'':<2} "
          f"{r['best_val_f1']:<10.4f} {r['dev_f1']:<10.4f}")
print(f"{'Ensemble (avg prob)':<20} {val_f1:<10.4f} {dev_f1:<10.4f}")

# Show config
print(f"\nConfig:")
print(f"  Verbalizer: {BEST_VERBALIZER_NAME}")
print(f"  Template [{BEST_TEMPLATE_IDX}]: \"{BEST_TEMPLATE.format(text='...', mask='[MASK]')}\"")
print(f"  lr: {LR:.2e}")
print(f"  weight_decay: {WEIGHT_DECAY:.2e}")
print(f"  head_lr_multiplier: {HEAD_LR_MULT}")
print(f"  seeds: {ENSEMBLE_SEEDS}")

In [ ]:
# Visualise ensemble agreement and probability distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Probability distributions per member
for i, (vp, r) in enumerate(zip(dev_probs_list, ensemble_results)):
    axes[0].hist(vp, bins=50, alpha=0.5, label=f"Member {i+1} (seed={r['seed']})")
axes[0].hist(dev_probs_ensemble, bins=50, alpha=0.7, color="black", histtype="step",
             linewidth=2, label="Ensemble")
axes[0].axvline(best_thresh, color="red", linestyle="--", label=f"Threshold={best_thresh:.3f}")
axes[0].set_xlabel("Probability")
axes[0].set_ylabel("Count")
axes[0].set_title("Dev Set Probability Distributions")
axes[0].legend(fontsize=8)

# 2. Member correlation scatter (member 1 vs 2)
axes[1].scatter(dev_probs_list[0], dev_probs_list[1], alpha=0.3, s=5)
axes[1].plot([0, 1], [0, 1], "r--", linewidth=1)
axes[1].set_xlabel(f"Member 1 (seed={ENSEMBLE_SEEDS[0]})")
axes[1].set_ylabel(f"Member 2 (seed={ENSEMBLE_SEEDS[1]})")
axes[1].set_title("Probability Correlation")
axes[1].set_aspect("equal")

# 3. Ensemble calibration: compare per-member dev F1 vs ensemble
member_f1s = [r["dev_f1"] for r in ensemble_results]
x_pos = list(range(len(member_f1s))) + [len(member_f1s)]
f1_vals = member_f1s + [dev_f1]
colours = ["steelblue"] * len(member_f1s) + ["darkorange"]
labels = [f"M{i+1}" for i in range(len(member_f1s))] + ["Ens"]
axes[2].bar(x_pos, f1_vals, color=colours, tick_label=labels)
axes[2].set_ylabel("Dev F1")
axes[2].set_title("Individual vs Ensemble F1")
axes[2].set_ylim(min(f1_vals) - 0.02, max(f1_vals) + 0.02)
for x, v in zip(x_pos, f1_vals):
    axes[2].text(x, v + 0.003, f"{v:.4f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/{EXP_NAME}_analysis.png", dpi=300)
plt.show()

Fixed HPs from Exp L:
  lr:               6.98e-06
  weight_decay:     6.92e-04
  head_lr_mult:     10
  warmup_fraction:  0.06
  label_smoothing:  0.0
  verbalizer:       true_false
  template_idx:     2
  ensemble seeds:   [42, 123, 7]



In [ ]:
# Save ensemble config and results
ensemble_config = {
    "base_experiment": "L_verbalizer",
    "seeds": ENSEMBLE_SEEDS,
    "best_params": BEST_PARAMS,
    "ensemble_threshold": best_thresh,
    "ensemble_val_f1": val_f1,
    "ensemble_dev_f1": float(dev_f1),
    "member_results": ensemble_results,
}
config_path = os.path.join(OUT_DIR, f"exp_{EXP_NAME}_config.json")
with open(config_path, "w") as f:
    json.dump(ensemble_config, f, indent=2)
LOG.info(f"Ensemble config saved to {config_path}")

2026-03-04 09:32:16,703 INFO:	============================================================
2026-03-04 09:32:16,704 INFO:	[P_verbalizer_ensemble] Training member 1/3 (seed=42)
2026-03-04 09:32:16,704 INFO:	============================================================


2026-03-04 09:32:16,703 INFO:	============================================================
2026-03-04 09:32:16,704 INFO:	[P_verbalizer_ensemble] Training member 1/3 (seed=42)
2026-03-04 09:32:16,704 INFO:	============================================================


2026-03-04 09:32:17,791 WARNING:	Sample 1307: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,796 WARNING:	Sample 1734: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,801 WARNING:	Sample 2274: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,812 WARNING:	Sample 3753: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,823 WARNING:	Sample 5179: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,827 WARNING:	Sample 5586: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,834 WARNING:	Sample 6606: [MASK] token was truncated (text may be too long for max_length=256). Falling back to posi

2026-03-04 09:32:16,703 INFO:	============================================================
2026-03-04 09:32:16,704 INFO:	[P_verbalizer_ensemble] Training member 1/3 (seed=42)
2026-03-04 09:32:16,704 INFO:	============================================================


2026-03-04 09:32:17,791 WARNING:	Sample 1307: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,796 WARNING:	Sample 1734: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,801 WARNING:	Sample 2274: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,812 WARNING:	Sample 3753: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,823 WARNING:	Sample 5179: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,827 WARNING:	Sample 5586: [MASK] token was truncated (text may be too long for max_length=256). Falling back to position 0.
2026-03-04 09:32:17,834 WARNING:	Sample 6606: [MASK] token was truncated (text may be too long for max_length=256). Falling back to posi


Member   Seed   Val F1     Dev F1     Dev P      Dev R     
------------------------------------------------------
1        42     0.5891     0.5991     0.5532     0.6533    
2        123    0.5902     0.5806     0.5735     0.5879    
3        7      0.5746     0.6154     0.5899     0.6432    
